# Exploratory Data Analysis (EDA)

Notebook ini melakukan eksplorasi awal terhadap tiga dataset diabetes yang digunakan
dalam penelitian *cross-dataset consistency*:

1. **PIDD** — Pima Indians Diabetes Dataset
2. **DPD** — Diabetes Prediction Dataset
3. **Early** — Early Stage Diabetes Risk Prediction Dataset

Tujuan EDA: memahami dimensi, distribusi kelas, *missing value*, dan karakteristik fitur
sebelum tahap *preprocessing* dan pelatihan model.

> **Catatan:** Jalankan `python download_data.py` terlebih dahulu (atau letakkan file CSV
> secara manual pada folder `datasets/`). Dataset yang belum tersedia akan dilewati.

In [ ]:
import sys
from pathlib import Path

# Pastikan root proyek ada di sys.path agar config & src dapat diimpor
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from preprocessing import load_dataset, split_feature_types

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 5)
print("Dataset terdaftar:", config.DATASET_ORDER)

In [ ]:
# Muat dataset yang tersedia
loaded = {}
for key in config.DATASET_ORDER:
    spec = config.DATASETS[key]
    path = Path(spec["path"])
    if not path.exists():
        print(f"[SKIP] {key}: file tidak ditemukan di {path}")
        continue
    try:
        X, y = load_dataset(key)
        loaded[key] = (X, y)
        print(f"[OK]   {key}: X={X.shape}, positif={int(y.sum())}/{len(y)} ({y.mean():.1%})")
    except Exception as e:
        print(f"[ERR]  {key}: {e}")

## 1. Ringkasan Dimensi dan Distribusi Kelas

In [ ]:
rows = []
for key, (X, y) in loaded.items():
    rows.append({
        "Dataset": config.DATASETS[key]["name"],
        "n_sampel": len(X),
        "n_fitur": X.shape[1],
        "positif": int(y.sum()),
        "negatif": int((y == 0).sum()),
        "rasio_positif": round(float(y.mean()), 4),
    })
summary = pd.DataFrame(rows)
summary

In [ ]:
if not summary.empty:
    ax = summary.set_index("Dataset")[["positif", "negatif"]].plot(kind="bar", stacked=True)
    ax.set_ylabel("Jumlah sampel")
    ax.set_title("Distribusi Kelas per Dataset")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 2. Analisis Missing Value

Khusus PIDD, nilai 0 pada kolom medis tertentu telah dikonversi menjadi `NaN` oleh
`load_dataset`, sehingga jumlah *missing value* di bawah ini mencerminkan kondisi nyata
yang akan ditangani oleh *mean imputation*.

In [ ]:
for key, (X, y) in loaded.items():
    miss = X.isna().sum()
    miss = miss[miss > 0]
    print(f"\n=== {config.DATASETS[key]['name']} ===")
    if miss.empty:
        print("Tidak ada missing value.")
    else:
        print(miss.to_string())

## 3. Statistik Deskriptif Fitur Numerik

In [ ]:
for key, (X, y) in loaded.items():
    num_cols, cat_cols = split_feature_types(X)
    print(f"\n=== {config.DATASETS[key]['name']} ===")
    print(f"Numerik ({len(num_cols)}): {num_cols}")
    print(f"Kategorikal ({len(cat_cols)}): {cat_cols}")
    if num_cols:
        display(X[num_cols].describe().T)

## 4. Distribusi Fitur Numerik (Histogram)

In [ ]:
for key, (X, y) in loaded.items():
    num_cols, _ = split_feature_types(X)
    if not num_cols:
        continue
    n = len(num_cols)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, col in enumerate(num_cols):
        axes[i].hist(X[col].dropna(), bins=30, color="#4C72B0")
        axes[i].set_title(col)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    fig.suptitle(config.DATASETS[key]["name"], y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Korelasi Antar Fitur Numerik

In [ ]:
for key, (X, y) in loaded.items():
    num_cols, _ = split_feature_types(X)
    if len(num_cols) < 2:
        continue
    corr = X[num_cols].corr()
    fig, ax = plt.subplots(figsize=(0.7 * len(num_cols) + 2, 0.7 * len(num_cols) + 2))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(num_cols)))
    ax.set_yticks(range(len(num_cols)))
    ax.set_xticklabels(num_cols, rotation=90)
    ax.set_yticklabels(num_cols)
    ax.set_title(f"Korelasi — {config.DATASETS[key]['name']}")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

## Kesimpulan EDA

- Ketiga dataset memiliki **dimensi dan tingkat ketidakseimbangan kelas yang berbeda**,
  yang memotivasi penggunaan **SMOTE** pada data latih dan **F1-Score** sebagai metrik utama.
- PIDD mengandung **missing value** (hasil konversi 0 → NaN) yang ditangani melalui
  *mean imputation*.
- Perbedaan karakteristik fitur antar dataset inilah yang menjadi dasar pengujian
  **cross-dataset consistency** pada tahap eksperimen utama (`run_experiment.py`).